# Memorando executável de pesquisa — identificação orientada a controle

**Projeto:** Data-Driven Dynamics and Control  
**Orientação:** Prof. Luciano A. Frezzato  
**Consolidação:** 18/09/2026

Este notebook é o registro técnico principal do projeto e substitui o antigo arquivo de descobertas. A intenção não é apenas registrar resultados finais, mas preservar a cadeia de raciocínio que levou de uma pergunta ampla sobre IA em controle até os experimentos atuais com Koopman, Lyapunov/LMIs, identificação entrada-saída, SINDy, observadores, EDPs e baterias.

> **Norte metodológico:** identificar modelos úteis para análise, estimação e controle, incorporando propriedades relevantes em vez de otimizar apenas erro de previsão.

Os scripts do repositório continuam sendo a implementação de referência. As células de código abaixo espelham os experimentos executados e geram as figuras usadas para interpretar cada etapa.

## 1. Como a pergunta de pesquisa evoluiu

### 1.1 Ponto de partida: IA + controle seguro
A formulação inicial combinava modelos aprendidos — Koopman, PINNs, redes neurais, Gaussian Processes — com síntese de controle por Lyapunov/LMI/MPC e camadas de segurança CBF/CLF.

A discussão de trabalhos com certificados neurais trouxe a pergunta crítica:

> **Quando o aprendizado realmente agrega valor em comparação com métodos clássicos que já fornecem estabilidade, robustez e segurança?**

Daí surgiram duas linhas:
1. **linha crítica:** IA versus LMI, Lyapunov, controle robusto e MPC;
2. **linha prioritária:** aprendizado para obter/refinar o modelo, mantendo controle e estimação estruturados.

A frase-síntese que guiou essa transição foi:

> **A IA não substitui o controlador; ela melhora o modelo que o controlador usa.**

### 1.2 Identificação orientada a controle
O survey de Sivaranjani et al. organizou a discussão em três maneiras de incorporar propriedades:
- parametrização por construção;
- soft constraints;
- hard constraints.

Isso motivou uma decisão metodológica importante com o Prof. Luciano: **não ir direto para Deep Koopman ou redes neurais**. Primeiro deveríamos entender a identificação clássica/estruturada, seus limites e as garantias que conseguimos impor.

## 2. Linha do tempo técnica

| Período | Experimento / decisão | Resultado conceitual |
|---|---|---|
| mar–jun/2026 | IA vs métodos clássicos; aprendizado online | aprendizado é mais defensável para modelagem/adaptação do que como substituição automática do controlador |
| 31/07/2026 | survey de identificação orientada a controle | reproduzir primeiro métodos estruturados |
| 08–14/08/2026 | Duffing + Koopman + RBF + mínimos quadrados | lifting permite regressão linear aproximada em observáveis |
| 14/08/2026 | reunião: ruído, I/O, LMI, SINDy, Hamiltonianos, instáveis | pesquisa incremental; GitHub como diário técnico |
| fim de ago/2026 | soft constraint, P=I, P variável + Schur | distinguir raio espectral, norma-2 e certificado Lyapunov |
| 31/08–01/09/2026 | Koopman entrada-saída com delays | boa saída não implica realização física correta do estado oculto |
| 01/09/2026 | Koopman em EDPs, Luenberger, baterias | redução de ordem e estimação passam ao centro |
| 17–18/09/2026 | SINDy + estado oculto + ruído/perturbação | identificabilidade e derivadas ruidosas são gargalos diferentes de predição |

## 3. Benchmark: sistema de Duffing modificado

O benchmark usado no repositório é

\[
\dot{x}_1=x_2
\]

\[
\dot{x}_2=-\delta x_2-x_1\cos(x_1+x_2)+u,\qquad \delta=2.
\]

A nomenclatura Duffing foi preservada por consistência com a referência/código reproduzidos, embora a não linearidade não seja a forma polinomial canônica do oscilador de Duffing.

### Configuração histórica
- passo de amostragem: dt = 0.01 s;
- 20.000 trajetórias independentes;
- 2 passos por trajetória;
- estados iniciais e entradas em [-1,1];
- teste a partir de x0 = [-0.6, 1.4];
- entrada de teste u(k)=0.8 sin(0.2k);
- thin-plate RBFs;
- a discussão começou com 8 RBFs; o código atual usa 10, refletindo a implementação de referência encontrada.

### Por que muitas trajetórias curtas?
Cada transição é uma amostra da regressão. Muitas trajetórias curtas independentes melhoram a cobertura do espaço de estados sem depender da evolução de uma trajetória longa. Em mínimos quadrados, porém, o peso é por **transição**, não por trajetória.

## 4. Koopman clássico — lifting e mínimos quadrados

Para o sistema não linear, escolhemos observáveis z = ψ(x) e identificamos

\[
z_{k+1}\approx Az_k+Bu_k.
\]

O operador de Koopman exato é, em geral, infinito-dimensional. A aproximação prática escolhe um subespaço finito. No experimento, o estado físico é mantido no começo do vetor lifted e são adicionadas thin-plate RBFs:

\[
z=[x_1,x_2,\phi_1(x),\ldots,\phi_N(x)]^T,
\quad
\phi_i=r_i^2\log(r_i+\varepsilon).
\]

Com dados empilhados,

\[
\Theta=[Z\;U],\qquad
\min_K \|\Theta K-Z_+\|_F^2.
\]

A matriz K é separada em A e B. Esse baseline é indispensável porque qualquer restrição de estabilidade precisa ser comparada contra o aumento de erro que ela produz.

### Métricas que passamos a separar
- erro one-step;
- free rollout;
- RMSE no espaço lifted;
- RMSE no estado físico;
- raio espectral ρ(A);
- norma espectral ||A||₂;
- custo computacional.

In [ ]:
# Duffing Oscillator System (as in SiShiAta 2026)
# dot{x1} = x2
# dot{x2} = -delta*x2 - x1*cos(x1 + x2) + u

import torch
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from torchdiffeq import odeint

# For reproducibility purposes
torch.manual_seed(156)

# Parameters of simulation
dt = 0.01 # Sampling time
T = 20.0 # Total simulation time
N = int(T/dt) # Number of time steps
t = torch.linspace(0, T, N + 1, dtype=torch.float64) # Time vector
method = 'rk4' # Integration method
options = {'step_size': dt} # Integration options

# Parameters of the Duffing oscillator
delta = 2.0 # Damping coefficient
x0 = -1.0 + 2.0 * torch.rand(2, dtype=torch.float64)
u0 = -1.0 + 2.0 * torch.rand((), dtype=torch.float64)

# Parameters of the thin plate, using Nrbf = 8, as in SiShiAta 2026
Nrbf = 10 # Number of radial basis functions -> although the article states 8 NrBF, the code uses 10
c = -1.0 + 2.0 * torch.rand((Nrbf, 2), dtype=torch.float64) # RBF centers in [-1, 1]^2

# Parameters of the training dataset
N_trajectories = 20000 # Number of independent trajectories
N_steps = 2 # Number of simulation steps per trajectory

# Duffing oscillator dynamics
def dynamics(t, x, u):
    x1, x2 = x
    dx1 = x2
    dx2 = -delta*x2 - x1*torch.cos(x1 + x2) + u
    return torch.stack([dx1, dx2])

# Simulation of one time step with constant input u
def simulate_step(x, u):
    t_step = torch.tensor([0.0, dt], dtype=torch.float64)
    solution = odeint(
        lambda t, x: dynamics(t, x, u),
        x,
        t_step,
        method=method,
        options=options
    )
    x_next = solution[-1]
    return x_next


# Training dataset
X = [] # Current states x(k)
U = [] # Inputs u(k)
X_next = [] # Next states x(k+1)


# Generation of the training dataset
for trajectory in range(N_trajectories):

    x = -1.0 + 2.0 * torch.rand(
        2,
        dtype=torch.float64
    ) # Random initial state in [-1, 1]^2
    for step in range(N_steps):
        u = -1.0 + 2.0 * torch.rand(
            (),
            dtype=torch.float64
        ) # Random input in [-1, 1]
        x_next = simulate_step(x, u)
        X.append(x)
        U.append(u)
        X_next.append(x_next)
        x = x_next

# Conversion of the training dataset to tensors
X = torch.stack(X)
U = torch.stack(U)
X_next = torch.stack(X_next)

# Creation of the Koopman lifted state using thin plate radial basis functions
def create_koopman_state(x, c, Nrbf):
    phi = []
    for i in range(Nrbf):
        r = torch.norm(x - c[i]) # Euclidean distance to RBF center
        phi_i = r**2 * torch.log(r + 1e-6) # Thin plate radial basis function
        phi.append(phi_i)

    phi = torch.stack(phi)
    z = torch.cat((x, phi))
    return z

# Creation of the Koopman lifted states
Z = []

for i in range(X.shape[0]):
    z = create_koopman_state(X[i], c, Nrbf)
    Z.append(z)
Z = torch.stack(Z)

# Creation of the next Koopman lifted states
Z_next = []
for i in range(X_next.shape[0]):
    z_next = create_koopman_state(X_next[i], c, Nrbf)
    Z_next.append(z_next)
Z_next = torch.stack(Z_next)

# Koopman with least squares regression
U = U.unsqueeze(1) # Reshape U to be a column vector
Theta = torch.cat(
    (Z, U),
    dim=1
)
# Least squares solution
K = torch.linalg.lstsq(
    Theta,
    Z_next
).solution

Nz = Z.shape[1]

A = K[:Nz, :].T # Koopman operator for the lifted state
B = K[Nz:, :].T # Koopman operator for the input


# Koopman with soft constraint
lambda_soft = 500.0 # Weight of the soft stability penalty -> SiShiAta 2026 uses 200

# Conversion from PyTorch to NumPy for CVXPY
Z_numpy = Z.numpy()
U_numpy = U.numpy()
Z_next_numpy = Z_next.numpy()

# Optimization variables
A_soft_variable = cp.Variable((Nz, Nz)) # TODO: investigate types in python, "symmetric", etc etc...
B_soft_variable = cp.Variable((Nz, 1))

gamma = cp.Variable()

# Prediction error
residual = Z_next_numpy - (
    Z_numpy @ A_soft_variable.T
    +
    U_numpy @ B_soft_variable.T
)

# LMI used to impose norm 2 for the step
I = np.eye(Nz)
lmi = cp.bmat([
    [gamma * I, A_soft_variable],
    [A_soft_variable.T, gamma * I]
])

# Soft stability constraints
constraints = [
    lmi >> 0
]

# Objective function
objective = cp.Minimize(
    cp.sum_squares(residual)
    +
    lambda_soft * cp.square(gamma)
)

# Optimization problem
problem = cp.Problem(
    objective,
    constraints
)

# Solve the optimization problem
problem.solve(
    solver=cp.SCS,
    verbose=True
)

# Optimized Koopman matrices
A_soft = torch.tensor(
    A_soft_variable.value,
    dtype=torch.float64
)

B_soft = torch.tensor(
    B_soft_variable.value,
    dtype=torch.float64
)


# Analysis of eigenvalues

# Unconstrained Koopman
eigenvalues = torch.linalg.eigvals(A)
rho_A = torch.max(
    torch.abs(eigenvalues)
)
# Soft-constrained Koopman
eigenvalues_soft = torch.linalg.eigvals(A_soft)
rho_A_soft = torch.max(
    torch.abs(eigenvalues_soft)
)

print("rho of A:", rho_A.item())
print("rho of A_soft:", rho_A_soft.item())
print("soft optimization status:", problem.status)

# Test against real system

# Test parameters
T_test = 10.0
N_test = int(T_test / dt)
t_test = torch.arange(
    N_test + 1,
    dtype=torch.float64
) * dt
x0_test = torch.tensor(
    [-0.6, 1.4],
    dtype=torch.float64
) # same as in SiShiAta 2026
# Test input
U_test = []
for k in range(N_test):
    u = 0.8 * torch.sin(
        torch.tensor(
            0.2 * k,
            dtype=torch.float64
        ) # same as in SiShiAta 2026
    )
    U_test.append(u)
U_test = torch.stack(U_test)

# Initial states
x_real = x0_test.clone()
z_koopman = create_koopman_state(
    x0_test,
    c,
    Nrbf
)
z_soft = create_koopman_state(
    x0_test,
    c,
    Nrbf
)

# Initialization of the trajectories
X_real_test = [x_real]
Z_koopman_test = [z_koopman]
Z_soft_test = [z_soft]

# Simulation of the three models
for k in range(N_test):
    u = U_test[k]
    # Real nonlinear system
    x_real = simulate_step(
        x_real,
        u
    )
    # Unconstrained Koopman model
    z_koopman = (
        A @ z_koopman
        +
        B[:, 0] * u
    )
    # Soft-constrained Koopman model
    z_soft = (
        A_soft @ z_soft
        +
        B_soft[:, 0] * u
    )
    X_real_test.append(x_real)
    Z_koopman_test.append(z_koopman)
    Z_soft_test.append(z_soft)


# Conversion of the test trajectories to tensors
X_real_test = torch.stack(X_real_test)
Z_koopman_test = torch.stack(Z_koopman_test)
Z_soft_test = torch.stack(Z_soft_test)

# Recovery of the original states
X_koopman_test = Z_koopman_test[:, :2]
X_soft_test = Z_soft_test[:, :2]

# Comparison of the trajectories

plt.figure(figsize=(12, 5))
# State x1
plt.subplot(1, 2, 1)
plt.plot(
    t_test.numpy(),
    X_real_test[:, 0].numpy(),
    label='Real system'
)
plt.plot(
    t_test.numpy(),
    X_koopman_test[:, 0].numpy(),
    '--',
    label='Koopman'
)
plt.plot(
    t_test.numpy(),
    X_soft_test[:, 0].numpy(),
    '--',
    label='Koopman soft'
)
plt.title('State x1')
plt.xlabel('Time [s]')
plt.ylabel('x1')
plt.grid()
plt.legend()

# State x2
plt.subplot(1, 2, 2)
plt.plot(
    t_test.numpy(),
    X_real_test[:, 1].numpy(),
    label='Real system'
)
plt.plot(
    t_test.numpy(),
    X_koopman_test[:, 1].numpy(),
    '--',
    label='Koopman'
)
plt.plot(
    t_test.numpy(),
    X_soft_test[:, 1].numpy(),
    '--',
    label='Koopman soft'
)
plt.title('State x2')
plt.xlabel('Time [s]')
plt.ylabel('x2')
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

## 5. Estabilidade do modelo Koopman: raio espectral, norma e Lyapunov

Para o modelo discreto autônomo z(k+1)=Az(k), a estabilidade assintótica exige ρ(A)<1.

Essa condição é difícil de impor diretamente como restrição convexa. Por isso investigamos três níveis.

### 5.1 Soft constraint por norma-2
A LMI

\[
\begin{bmatrix}
\gamma I & A\\
A^T & \gamma I
\end{bmatrix}\succeq 0
\]

impõe ||A||₂ ≤ γ. No experimento, γ é penalizado no objetivo.

**Importante:** isso só vira garantia de estabilidade se o resultado satisfizer γ<1 ou se γ≤γmax<1 for imposto explicitamente. Caso contrário, é regularização, não uma prova dura.

### 5.2 P = I
Com V(z)=zᵀz,

\[
A^TA-I\prec0.
\]

Pelo complemento de Schur,

\[
\begin{bmatrix}
I&A\\
A^T&I
\end{bmatrix}\succ0.
\]

É convexa em A, simples e útil como primeiro teste, mas conservadora porque exige contração na métrica Euclidiana.

### 5.3 P variável
A condição geral é

\[
P\succ0,\qquad A^TPA-P\prec0.
\]

Como A e P aparecem multiplicados, a formulação é bilinear. A substituição discutida e implementada foi

\[
Y=PA,\qquad G=PB,
\]

recuperando depois A=P⁻¹Y e B=P⁻¹G. A LMI de Schur passa a ser

\[
\begin{bmatrix}
P&Y^T\\
Y&P
\end{bmatrix}\succ0.
\]

Foi adicionada a normalização tr(P)=nz para remover ambiguidade de escala.

### Sutileza que não deve se perder
Para manter convexidade, a função objetivo de P variável é escrita sobre o residual transformado PZ₊−YZ−GU. Isso **não é exatamente o mesmo objetivo LS** do baseline; ele pondera o erro pela métrica P. Comparações devem reconhecer essa diferença.

In [ ]:
# Duffing Oscillator System (as in SiShiAta 2026)
# dot{x1} = x2
# dot{x2} = -delta*x2 - x1*cos(x1 + x2) + u

import time

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchdiffeq import odeint

# For reproducibility purposes
torch.manual_seed(156)

# Parameters of simulation
dt = 0.01
T = 20.0
N = int(T / dt)
t = torch.linspace(0, T, N + 1, dtype=torch.float64)
method = "rk4"
options = {"step_size": dt}

# Parameters of the Duffing oscillator
delta = 2.0
x0 = -1.0 + 2.0 * torch.rand(2, dtype=torch.float64)
u0 = -1.0 + 2.0 * torch.rand((), dtype=torch.float64)

# Parameters of the thin plate RBFs
Nrbf = 10
c = -1.0 + 2.0 * torch.rand((Nrbf, 2), dtype=torch.float64)

# Parameters of the training dataset
N_trajectories = 20000
N_steps = 2

# Weight used in the soft constraint
lambda_soft = 500.0

# Numerical margins for the hard LMIs
LMI_EPS = 1e-6
P_MIN_EIG = 1e-3

PARITY_CHECK_TRAJECTORIES = 5
PARITY_TOL = 1e-12


# Duffing oscillator dynamics
def dynamics(t, x, u):
    x1 = x[..., 0]
    x2 = x[..., 1]

    dx1 = x2
    dx2 = -delta * x2 - x1 * torch.cos(x1 + x2) + u

    return torch.stack((dx1, dx2), dim=-1)


# Simulation of one time step with constant input u
def simulate_step(x, u):
    t_step = torch.tensor([0.0, dt], dtype=torch.float64)
    solution = odeint(
        lambda current_t, current_x: dynamics(current_t, current_x, u),
        x,
        t_step,
        method=method,
        options=options,
    )
    return solution[-1]


# Creation of the Koopman lifted state using thin plate radial basis functions
def create_koopman_state(x, centers, n_rbf):
    phi = []
    for i in range(n_rbf):
        r = torch.norm(x - centers[i])
        phi_i = r**2 * torch.log(r + 1e-6)
        phi.append(phi_i)

    phi = torch.stack(phi)
    return torch.cat((x, phi))


def lift_batch(X_batch):
    diff = X_batch[:, None, :] - c[None, :, :]
    r = torch.norm(diff, dim=2)
    phi = r**2 * torch.log(r + 1e-6)
    return torch.cat((X_batch, phi), dim=1)


# Generation of the training random variables
def draw_training_random_variables():
    initial_states = torch.empty((N_trajectories, 2), dtype=torch.float64)
    inputs = torch.empty((N_trajectories, N_steps), dtype=torch.float64)

    for trajectory in range(N_trajectories):
        initial_states[trajectory] = -1.0 + 2.0 * torch.rand(
            2,
            dtype=torch.float64,
        )
        for step in range(N_steps):
            inputs[trajectory, step] = -1.0 + 2.0 * torch.rand(
                (),
                dtype=torch.float64,
            )

    return initial_states, inputs


# Check that batched integration reproduces the scalar implementation
def verify_batched_odeint_parity(initial_states, inputs):
    n_check = min(PARITY_CHECK_TRAJECTORIES, initial_states.shape[0])

    x_batch = initial_states[:n_check].clone()
    x_scalar = initial_states[:n_check].clone()
    max_error = 0.0

    for step in range(N_steps):
        x_batch = simulate_step(x_batch, inputs[:n_check, step])

        scalar_next = []
        for trajectory in range(n_check):
            scalar_next.append(
                simulate_step(
                    x_scalar[trajectory],
                    inputs[trajectory, step],
                )
            )
        x_scalar = torch.stack(scalar_next)

        max_error = max(
            max_error,
            torch.max(torch.abs(x_batch - x_scalar)).item(),
        )

    print(f"torchdiffeq batch/scalar parity max error: {max_error:.3e}")

    if max_error > PARITY_TOL:
        raise RuntimeError(
            "Batched torchdiffeq does not reproduce the scalar implementation: "
            f"{max_error:.3e} > {PARITY_TOL:.1e}."
        )


# Training dataset
def generate_training_dataset():
    initial_states, inputs = draw_training_random_variables()
    verify_batched_odeint_parity(initial_states, inputs)

    x = initial_states.clone()

    X = torch.empty((N_trajectories, N_steps, 2), dtype=torch.float64)
    U = torch.empty((N_trajectories, N_steps, 1), dtype=torch.float64)
    X_next = torch.empty_like(X)

    for step in range(N_steps):
        u = inputs[:, step]
        x_next = simulate_step(x, u)

        X[:, step, :] = x
        U[:, step, 0] = u
        X_next[:, step, :] = x_next

        x = x_next

    X = X.reshape(-1, 2)
    U = U.reshape(-1, 1)
    X_next = X_next.reshape(-1, 2)

    Z = lift_batch(X)
    Z_next = lift_batch(X_next)

    return X, U, X_next, Z, Z_next


# Gram matrix used to reduce the CVXPY problem size
def psd_gram(data, normalize=True):
    gram = data.T @ data
    if normalize:
        gram = gram / data.shape[0]
    gram = 0.5 * (gram + gram.T)
    return cp.psd_wrap(gram)


def ls_quadratic_expression(A_var, B_var, Z_np, U_np, Z_next_np, normalize=True):
    n = Z_np.shape[0]
    theta = np.hstack((Z_np, U_np))

    scale = n if normalize else 1.0
    gram_theta = psd_gram(theta, normalize=normalize)
    cross = (theta.T @ Z_next_np) / scale
    constant = np.sum(Z_next_np**2) / scale

    M = cp.hstack((A_var, B_var))

    terms = []
    for output_index in range(Z_next_np.shape[1]):
        row = M[output_index, :]
        terms.append(
            cp.quad_form(row, gram_theta)
            - 2.0 * cross[:, output_index] @ row
        )

    return cp.sum(terms) + constant


# Koopman with soft stability constraint
def identify_koopman_soft(Z, U, Z_next):
    Z_np = Z.numpy()
    U_np = U.numpy()
    Z_next_np = Z_next.numpy()

    nz = Z.shape[1]
    I = np.eye(nz)

    A_var = cp.Variable((nz, nz))
    B_var = cp.Variable((nz, 1))
    gamma = cp.Variable()

    residual_sum_squares = ls_quadratic_expression(
        A_var,
        B_var,
        Z_np,
        U_np,
        Z_next_np,
        normalize=False,
    )

    # LMI used to impose the spectral norm bound ||A||_2 <= gamma
    soft_lmi = cp.bmat(
        [
            [gamma * I, A_var],
            [A_var.T, gamma * I],
        ]
    )

    problem = cp.Problem(
        cp.Minimize(
            residual_sum_squares
            + lambda_soft * cp.square(gamma)
        ),
        [soft_lmi >> 0],
    )

    problem.solve(
        solver=cp.SCS,
        verbose=False,
    )

    if A_var.value is None or B_var.value is None or gamma.value is None:
        raise RuntimeError(
            f"Soft identification failed. Solver status: {problem.status}"
        )

    A = torch.tensor(A_var.value, dtype=torch.float64)
    B = torch.tensor(B_var.value, dtype=torch.float64)

    return A, B, float(gamma.value), problem.status, problem.value


# Koopman with hard stability constraint using P = I
def identify_koopman_p_identity(Z, U, Z_next):
    Z_np = Z.numpy()
    U_np = U.numpy()
    Z_next_np = Z_next.numpy()

    nz = Z.shape[1]
    I = np.eye(nz)

    A_var = cp.Variable((nz, nz))
    B_var = cp.Variable((nz, 1))

    mse_objective = ls_quadratic_expression(
        A_var,
        B_var,
        Z_np,
        U_np,
        Z_next_np,
        normalize=True,
    )

    # Equivalent to A.T @ A - I < 0
    lyapunov_lmi = cp.bmat(
        [
            [I, A_var],
            [A_var.T, I],
        ]
    )

    problem = cp.Problem(
        cp.Minimize(mse_objective),
        [lyapunov_lmi >> LMI_EPS * np.eye(2 * nz)],
    )

    problem.solve(
        solver=cp.SCS,
        verbose=False,
        eps=1e-5,
        max_iters=15000,
        warm_start=True,
    )

    if A_var.value is None or B_var.value is None:
        raise RuntimeError(
            f"P=I identification failed. Solver status: {problem.status}"
        )

    A = torch.tensor(A_var.value, dtype=torch.float64)
    B = torch.tensor(B_var.value, dtype=torch.float64)

    return A, B, problem.status, problem.value


# Koopman with variable Lyapunov matrix P
def identify_koopman_variable_p(Z, U, Z_next):
    Z_np = Z.numpy()
    U_np = U.numpy()
    Z_next_np = Z_next.numpy()

    nz = Z.shape[1]
    I = np.eye(nz)

    # Variable substitution: Y = P A and G = P B
    D = np.hstack((Z_next_np, Z_np, U_np))
    gram_D = psd_gram(D, normalize=True)

    P = cp.Variable((nz, nz), symmetric=True)
    Y = cp.Variable((nz, nz))
    G = cp.Variable((nz, 1))

    C = cp.vstack((P, -Y.T, -G.T))

    transformed_mse = cp.sum(
        [
            cp.quad_form(C[:, output_index], gram_D)
            for output_index in range(nz)
        ]
    )

    # Schur-complement form of A.T @ P @ A - P < 0
    schur_lmi = cp.bmat(
        [
            [P, Y.T],
            [Y, P],
        ]
    )

    constraints = [
        P >> P_MIN_EIG * I,
        cp.trace(P) == nz,
        schur_lmi >> LMI_EPS * np.eye(2 * nz),
    ]

    problem = cp.Problem(
        cp.Minimize(transformed_mse),
        constraints,
    )

    problem.solve(
        solver=cp.SCS,
        verbose=False,
        eps=1e-5,
        max_iters=15000,
        warm_start=True,
    )

    if P.value is None or Y.value is None or G.value is None:
        raise RuntimeError(
            f"Variable-P identification failed. Solver status: {problem.status}"
        )

    A_np = np.linalg.solve(P.value, Y.value)
    B_np = np.linalg.solve(P.value, G.value)

    A = torch.tensor(A_np, dtype=torch.float64)
    B = torch.tensor(B_np, dtype=torch.float64)
    P_np = (
        P.value.toarray()
        if hasattr(P.value, "toarray")
        else np.asarray(P.value)
    )
    
    P_torch = torch.tensor(P_np, dtype=torch.float64)
    return A, B, P_torch, problem.status, problem.value


# Analysis functions
def spectral_radius(A):
    return torch.max(torch.abs(torch.linalg.eigvals(A))).item()


def spectral_norm(A):
    return torch.linalg.matrix_norm(A, ord=2).item()


def lyapunov_max_eigenvalue(A, P):
    M = A.T @ P @ A - P
    M = 0.5 * (M + M.T)
    return torch.max(torch.linalg.eigvalsh(M)).item()


def one_step_metrics(A, B, Z, U, Z_next, X_next):
    Z_pred = Z @ A.T + U @ B.T

    lifted_rmse = torch.sqrt(
        torch.mean((Z_pred - Z_next) ** 2)
    ).item()

    state_rmse = torch.sqrt(
        torch.mean((Z_pred[:, :2] - X_next) ** 2)
    ).item()

    return lifted_rmse, state_rmse


def simulate_identified_model(A, B, x0_test, U_test):
    z = create_koopman_state(x0_test, c, Nrbf)
    trajectory = [z]

    for u in U_test:
        z = A @ z + B[:, 0] * u
        trajectory.append(z)

    return torch.stack(trajectory)


def rollout_rmse(Z_model, X_real):
    X_model = Z_model[:, :2]
    return torch.sqrt(
        torch.mean((X_model - X_real) ** 2)
    ).item()


# Test against the real system
def build_test_trajectory():
    T_test = 10.0
    N_test = int(T_test / dt)
    t_test = torch.arange(
        N_test + 1,
        dtype=torch.float64,
    ) * dt

    x0_test = torch.tensor(
        [-0.6, 1.4],
        dtype=torch.float64,
    )

    U_test = []
    for k in range(N_test):
        u = 0.8 * torch.sin(
            torch.tensor(
                0.2 * k,
                dtype=torch.float64,
            )
        )
        U_test.append(u)
    U_test = torch.stack(U_test)

    x_real = x0_test.clone()
    X_real = [x_real]

    for u in U_test:
        x_real = simulate_step(x_real, u)
        X_real.append(x_real)

    return t_test, x0_test, U_test, torch.stack(X_real)


# Comparison table
def print_comparison_table(results):
    print("\n" + "=" * 114)
    print("KOOPMAN IDENTIFICATION - SAME DATA / SAME TEST / THREE IDENTIFICATION METHODS")
    print("=" * 114)
    print(
        f"{'Model':<31}"
        f"{'rho(A)':>12}"
        f"{'||A||2':>12}"
        f"{'1-step z RMSE':>18}"
        f"{'1-step x RMSE':>18}"
        f"{'rollout x RMSE':>18}"
    )
    print("-" * 114)

    for result in results:
        print(
            f"{result['name']:<31}"
            f"{result['rho']:>12.6f}"
            f"{result['norm2']:>12.6f}"
            f"{result['one_step_z']:>18.6e}"
            f"{result['one_step_x']:>18.6e}"
            f"{result['rollout_x']:>18.6e}"
        )

    print("=" * 114)


# Comparison of the trajectories
def plot_comparison(t_test, X_real, model_trajectories):
    styles = {
        "Koopman soft": {
            "linestyle": (0, (6, 2, 1, 2)),
            "linewidth": 2.0,
            "marker": "o",
            "markevery": 100,
            "markersize": 3.5,
        },
        "Koopman P=I": {
            "linestyle": "-.",
            "linewidth": 1.8,
            "marker": "s",
            "markevery": 100,
            "markersize": 3.5,
        },
        "Koopman variable P": {
            "linestyle": ":",
            "linewidth": 2.2,
            "marker": "x",
            "markevery": 100,
            "markersize": 4.0,
        },
    }

    plt.figure(figsize=(13, 5))

    for subplot_index, state_index in enumerate((0, 1), start=1):
        plt.subplot(1, 2, subplot_index)
        plt.plot(
            t_test.numpy(),
            X_real[:, state_index].numpy(),
            linestyle="-",
            linewidth=2.6,
            label="Real system",
        )

        for label, Z_model in model_trajectories:
            plt.plot(
                t_test.numpy(),
                Z_model[:, state_index].numpy(),
                label=label,
                **styles[label],
            )

        plt.title(f"State x{state_index + 1}")
        plt.xlabel("Time [s]")
        plt.ylabel(f"x{state_index + 1}")
        plt.grid()
        plt.legend()

    plt.tight_layout()
    plt.show()


# Main experiment
def main():
    total_start = time.perf_counter()

    stage_start = time.perf_counter()
    print("Generating Duffing dataset with torchdiffeq RK4...")
    X, U, X_next, Z, Z_next = generate_training_dataset()
    print(f"  done in {time.perf_counter() - stage_start:.3f} s")

    nz = Z.shape[1]
    print(f"Samples: {Z.shape[0]}")
    print(f"Lifted-state dimension: {nz}")

    stage_start = time.perf_counter()
    print("\n[1/3] Koopman soft constraint (lambda=500)...")
    A_soft, B_soft, gamma_soft, status_soft, objective_soft = (
        identify_koopman_soft(Z, U, Z_next)
    )
    print(f"  done in {time.perf_counter() - stage_start:.3f} s")

    stage_start = time.perf_counter()
    print("[2/3] Koopman with hard Lyapunov condition, P = I...")
    A_identity, B_identity, status_identity, objective_identity = (
        identify_koopman_p_identity(Z, U, Z_next)
    )
    print(f"  done in {time.perf_counter() - stage_start:.3f} s")

    stage_start = time.perf_counter()
    print("[3/3] Koopman with variable P + Schur/Lyapunov LMI...")
    A_p, B_p, P_p, status_p, objective_p = identify_koopman_variable_p(
        Z,
        U,
        Z_next,
    )
    print(f"  done in {time.perf_counter() - stage_start:.3f} s")

    t_test, x0_test, U_test, X_real = build_test_trajectory()

    Z_soft = simulate_identified_model(A_soft, B_soft, x0_test, U_test)
    Z_identity = simulate_identified_model(
        A_identity,
        B_identity,
        x0_test,
        U_test,
    )
    Z_p = simulate_identified_model(A_p, B_p, x0_test, U_test)

    models = [
        ("Koopman soft", A_soft, B_soft, Z_soft),
        ("Koopman P=I", A_identity, B_identity, Z_identity),
        ("Koopman variable P", A_p, B_p, Z_p),
    ]

    results = []
    for name, A_model, B_model, Z_rollout in models:
        one_step_z, one_step_x = one_step_metrics(
            A_model,
            B_model,
            Z,
            U,
            Z_next,
            X_next,
        )

        results.append(
            {
                "name": name,
                "rho": spectral_radius(A_model),
                "norm2": spectral_norm(A_model),
                "one_step_z": one_step_z,
                "one_step_x": one_step_x,
                "rollout_x": rollout_rmse(Z_rollout, X_real),
            }
        )

    print_comparison_table(results)

    I_torch = torch.eye(nz, dtype=torch.float64)
    identity_lyapunov_eig = lyapunov_max_eigenvalue(
        A_identity,
        I_torch,
    )
    variable_p_lyapunov_eig = lyapunov_max_eigenvalue(
        A_p,
        P_p,
    )

    p_eigs = torch.linalg.eigvalsh(P_p)
    p_condition_number = (
        torch.max(p_eigs) / torch.min(p_eigs)
    ).item()

    print("\nConstraint / solver verification")
    print("--------------------------------")
    print(f"Soft solver status:                {status_soft}")
    print(f"Soft objective:                    {objective_soft:.6e}")
    print(f"Soft gamma:                        {gamma_soft:.6e}")
    print(f"Soft ||A||2:                       {spectral_norm(A_soft):.6e}")
    print(f"Soft gamma-||A||2:                 {gamma_soft - spectral_norm(A_soft):.6e}")
    print()
    print(f"P=I solver status:                 {status_identity}")
    print(f"P=I optimization objective:        {objective_identity:.6e}")
    print(
        "max eig(A^T A - I):            "
        f"{identity_lyapunov_eig:.6e}"
    )
    print()
    print(f"Variable-P solver status:          {status_p}")
    print(f"Variable-P optimization objective: {objective_p:.6e}")
    print(
        "max eig(A^T P A - P):          "
        f"{variable_p_lyapunov_eig:.6e}"
    )
    print(
        "eig(P) range:                   "
        f"[{torch.min(p_eigs).item():.6e}, "
        f"{torch.max(p_eigs).item():.6e}]"
    )
    print(f"cond(P) from eigenvalues:          {p_condition_number:.6e}")

    print(f"\nTotal runtime: {time.perf_counter() - total_start:.3f} s")

    plot_comparison(
        t_test,
        X_real,
        [
            ("Koopman soft", Z_soft),
            ("Koopman P=I", Z_identity),
            ("Koopman variable P", Z_p),
        ],
    )


if __name__ == "__main__":
    main()


## 6. O que a etapa de Lyapunov/LMI ensinou

1. Erro local pequeno não garante bom rollout.
2. ρ(A) e ||A||₂ não são equivalentes; ||A||₂<1 é uma condição suficiente mais forte.
3. P=I é um ótimo baseline didático, mas pode ser conservador.
4. P variável permite uma métrica elipsoidal adequada à dinâmica, ao custo de uma formulação mais delicada.
5. A LMI certifica o **modelo linear lifted identificado**. Isso não é automaticamente uma prova de estabilidade global da planta não linear.
6. Se a planta verdadeira é instável em malha aberta, forçar A estável durante a identificação pode produzir um modelo incorreto.

Esse sexto ponto levou à discussão de sistemas como o pêndulo invertido: usar experimentos curtos ou dados em malha fechada pode ser necessário para coletar informação antes de a trajetória escapar da região útil.

## 7. Identificação entrada-saída: quando o estado não é medido

A pergunta do orientador foi diretamente ligada à aplicação em baterias:

> **O que acontece quando só temos entrada u e saída y?**

No Duffing escolhemos y=x₂ e construímos um estado de delays

\[
\zeta_k=[y_k,\;y_{k-1},\;u_{k-1}]^T.
\]

Depois fazemos lifting de ζ com RBFs e identificamos um modelo linear no espaço aumentado.

Dois testes precisam ser separados:
- **one-step:** usa histórico verdadeiro a cada passo;
- **free rollout:** depois da inicialização, o modelo realimenta a própria previsão.

Essa distinção foi central: modelos com one-step excelente podem acumular erro no rollout.

In [ ]:
# Duffing Oscillator - Input/Output Koopman identification
# dot{x1} = x2
# dot{x2} = -delta*x2 - x1*cos(x1 + x2) + u
# y = x2

import matplotlib.pyplot as plt
import torch

# For reproducibility purposes
torch.set_default_dtype(torch.float64)
torch.manual_seed(156)

# Parameters of simulation
dt = 0.01
delta = 2.0

# Parameters of the training dataset
N_trajectories = 20000
N_steps = 2
Nrbf = 10

# Output matrix: y = x2
Cy = torch.tensor([0.0, 1.0])


# Duffing oscillator dynamics, used only to simulate the plant
def dynamics(x, u):
    x1 = x[..., 0]
    x2 = x[..., 1]

    dx1 = x2
    dx2 = -delta * x2 - x1 * torch.cos(x1 + x2) + u

    return torch.stack([dx1, dx2], dim=-1)


# Simulation of one time step using RK4
def simulate_step(x, u):
    k1 = dynamics(x, u)
    k2 = dynamics(x + 0.5 * dt * k1, u)
    k3 = dynamics(x + 0.5 * dt * k2, u)
    k4 = dynamics(x + dt * k3, u)

    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


# Output of the Duffing oscillator
def output(x):
    return x @ Cy


# Input/output training dataset
# zeta_k = [y_k, y_(k-1), u_(k-1)]
x0 = -1.0 + 2.0 * torch.rand((N_trajectories, 2))
u0 = -1.0 + 2.0 * torch.rand(N_trajectories)
u1 = -1.0 + 2.0 * torch.rand(N_trajectories)

y0 = output(x0)

x1 = simulate_step(x0, u0)
y1 = output(x1)

x2 = simulate_step(x1, u1)
y2 = output(x2)

# Delay states at k = 1 and k = 2
ZETA = torch.stack([y1, y0, u0], dim=1)
ZETA_next = torch.stack([y2, y1, u1], dim=1)

# Input associated with the transition ZETA -> ZETA_next
U = u1.unsqueeze(1)

n_zeta = ZETA.shape[1]


# Koopman lifting of the input/output delay state
rbf_centers = -1.0 + 2.0 * torch.rand((Nrbf, n_zeta))


def create_koopman_state(zeta):
    diff = zeta.unsqueeze(-2) - rbf_centers
    r = torch.linalg.vector_norm(diff, dim=-1)
    phi = r**2 * torch.log(r + 1e-6)

    return torch.cat([zeta, phi], dim=-1)


Z = create_koopman_state(ZETA)
Z_next = create_koopman_state(ZETA_next)


# Koopman with least squares regression
Theta = torch.cat([Z, U], dim=1)

K = torch.linalg.lstsq(
    Theta,
    Z_next,
).solution

Nz = Z.shape[1]

A = K[:Nz, :].T
B = K[Nz:, :].T


# Training diagnostics
Z_next_hat_train = Z @ A.T + U @ B.T
Y_next_hat_train = Z_next_hat_train[:, 0]
Y_next_train = ZETA_next[:, 0]

train_error = Y_next_hat_train - Y_next_train
train_mae = torch.mean(torch.abs(train_error))
train_rmse = torch.sqrt(torch.mean(train_error**2))

rho_A = torch.max(torch.abs(torch.linalg.eigvals(A)))


# Test against the real system
T_test = 10.0
N_test = int(T_test / dt)
t_test = torch.arange(N_test + 1) * dt

x0_test = torch.tensor([-0.6, 1.4])

U_test = 0.8 * torch.sin(
    torch.arange(N_test) * 0.2
)

x_true = x0_test.clone()
Y_true = [output(x_true)]

for k in range(N_test):
    x_true = simulate_step(x_true, U_test[k])
    Y_true.append(output(x_true))

Y_true = torch.stack(Y_true)


# One-step-ahead prediction using the real input/output history
Y_one_step = [Y_true[0], Y_true[1]]

for k in range(1, N_test):
    zeta_true = torch.stack([
        Y_true[k],
        Y_true[k - 1],
        U_test[k - 1],
    ])

    z_true = create_koopman_state(zeta_true)
    z_next = A @ z_true + B[:, 0] * U_test[k]
    Y_one_step.append(z_next[0])

Y_one_step = torch.stack(Y_one_step)


# Free rollout using one measured transition for initialization
zeta_initial = torch.stack([
    Y_true[1],
    Y_true[0],
    U_test[0],
])

z_koopman = create_koopman_state(zeta_initial)
Y_rollout = [Y_true[0], Y_true[1]]

for k in range(1, N_test):
    z_koopman = A @ z_koopman + B[:, 0] * U_test[k]
    Y_rollout.append(z_koopman[0])

Y_rollout = torch.stack(Y_rollout)


# Error metrics
def metrics(y_true, y_hat):
    error = y_hat - y_true

    mae = torch.mean(torch.abs(error))
    rmse = torch.sqrt(torch.mean(error**2))
    iae = dt * torch.sum(torch.abs(error))

    return mae.item(), rmse.item(), iae.item()


one_mae, one_rmse, one_iae = metrics(
    Y_true,
    Y_one_step,
)

roll_mae, roll_rmse, roll_iae = metrics(
    Y_true,
    Y_rollout,
)


# Results
print("\n" + "=" * 78)
print("DUFFING - INPUT/OUTPUT KOOPMAN IDENTIFICATION")
print("=" * 78)
print("Measured output: y = x2")
print("Delay state: zeta_k = [y_k, y_(k-1), u_(k-1)]")
print(f"Training trajectories: {N_trajectories}")
print(f"Delay-state dimension: {n_zeta}")
print(f"Lifted dimension: {Nz} = {n_zeta} raw I/O coordinates + {Nrbf} RBFs")

print("\nTraining - one-step output fit")
print(f"  MAE : {train_mae.item():.8f}")
print(f"  RMSE: {train_rmse.item():.8f}")

print("\nSpectral radius")
print(f"  rho(A): {rho_A.item():.8f}")

print("\nTest - one-step ahead")
print(f"  MAE={one_mae:.8f} | RMSE={one_rmse:.8f} | IAE={one_iae:.8f}")

print("\nTest - free rollout")
print(f"  MAE={roll_mae:.8f} | RMSE={roll_rmse:.8f} | IAE={roll_iae:.8f}")
print("=" * 78)


# Comparison of the output trajectories
plt.figure(figsize=(12, 5))
plt.plot(
    t_test.numpy(),
    Y_true.numpy(),
    label="Real system",
)
plt.plot(
    t_test.numpy(),
    Y_rollout.numpy(),
    "--",
    label="Koopman I/O",
)
plt.title("Output y = x2")
plt.xlabel("Time [s]")
plt.ylabel("y")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

## 8. A descoberta conceitual da etapa I/O: previsão ≠ realização física

O estado interno de uma realização entrada-saída não é único. Um modelo de delays/Koopman pode prever muito bem y sem que nenhuma coordenada corresponda exatamente ao estado físico oculto x₁.

Portanto:

> **prever bem a saída não garante reconstruir corretamente o estado físico interno.**

Isso muda o papel de um observador. Para

\[
z_{k+1}=Az_k+Bu_k,\qquad y_k=Cz_k,
\]

um Luenberger pode ser escrito como

\[
\hat z_{k+1}=A\hat z_k+Bu_k+L(y_k-C\hat z_k),
\]

com erro

\[
e_{k+1}=(A-LC)e_k.
\]

Se o par é detectável/observável, podemos estabilizar o erro do **estado lifted**. Mas, se z é apenas uma coordenada abstrata de delays, isso não garante que tenhamos estimado uma variável física como concentração, SOC ou posição.

Para recuperar significado físico precisamos ancorar a realização com:
- estados físicos incluídos explicitamente no lifting;
- relações físicas conhecidas;
- decoder calibrado com estado;
- medições adicionais;
- restrições estruturais.

## 9. Mudança de direção: física parcial + SINDy

Com y=x₂, sabemos pela física que

\[
\dot x_1=x_2=y,
\]

então

\[
x_1(t)=x_1(0)+\int_0^t y(\tau)\,d\tau.
\]

A forma temporal de x₁ pode ser reconstruída a partir da saída, mas o offset x₁(0) continua livre.

Isso motivou uma estratégia grey-box:
1. a física conhecida fixa o significado de x₁;
2. testamos candidatos para x₁(0);
3. reconstruímos x₁ por integração;
4. usamos SINDy para identificar a equação desconhecida de \dot y;
5. escolhemos o candidato que melhor explica os dados.

Aqui o SINDy não é apenas uma ferramenta de descoberta de equações; ele atua como identificador esparso condicionado por física parcial.

## 10. SINDy, biblioteca e STLSQ

A forma geral é

\[
\dot y=\Theta(x_1,y,u)\xi.
\]

A structured library contém o termo verdadeiro x₁ cos(x₁+y). A blind library remove esse termo e oferece funções alternativas, permitindo testar o efeito de uma hipótese estrutural incompleta.

O STLSQ usado no projeto:
1. resolve mínimos quadrados;
2. remove coeficientes abaixo do threshold;
3. refaz LS apenas nos termos ativos;
4. repete.

O objetivo é obter uma equação parcimoniosa, evitando que dezenas de termos pequenos absorvam ruído ou erro de modelagem.

In [ ]:
# Duffing Oscillator - SINDy with hidden-state reconstruction
# dot{x1} = x2
# dot{x2} = -delta*x2 - x1*cos(x1 + x2) + u
# y = x2

import matplotlib.pyplot as plt
import torch

# For reproducibility purposes
torch.set_default_dtype(torch.float64)
torch.manual_seed(156)

# Parameters of simulation
dt = 0.01
T = 10.0
N = int(T / dt)
delta = 2.0
threshold = 0.1

# Duffing oscillator dynamics, used only to simulate the plant
def dynamics(x, u):
    x1, x2 = x
    return torch.stack([
        x2,
        -delta * x2 - x1 * torch.cos(x1 + x2) + u,
    ])

# Simulation of one time step using RK4
def simulate_step(x, u):
    k1 = dynamics(x, u)
    k2 = dynamics(x + 0.5 * dt * k1, u)
    k3 = dynamics(x + 0.5 * dt * k2, u)
    k4 = dynamics(x + dt * k3, u)
    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

# Reconstruction from the known physics: dot{x1} = y
def reconstruct_x1(y, x10):
    x1 = torch.empty_like(y)
    x1[0] = x10
    x1[1:] = x10 + dt * torch.cumsum(
        0.5 * (y[:-1] + y[1:]),
        dim=0,
    )
    return x1

# SINDy candidate library
def library(x1, y, u):
    s = x1 + y
    return torch.column_stack([
        torch.ones_like(y),
        x1,
        y,
        u,
        torch.cos(s),
        x1 * torch.cos(s),
    ])

# Sequential thresholded least squares
def sindy(theta, target):
    xi = torch.linalg.lstsq(theta, target).solution
    for _ in range(10):
        active = torch.abs(xi) >= threshold
        xi.zero_()
        xi[active] = torch.linalg.lstsq(theta[:, active], target).solution
    return xi

# Real trajectory
x0 = torch.tensor([-0.6, 1.4])
U = 0.8 * torch.sin(torch.arange(N) * 0.2)

X_true = [x0]
for u in U:
    X_true.append(simulate_step(X_true[-1], u))
X_true = torch.stack(X_true)

# Measurements available to the identifier
Y = X_true[:, 1]
dY = (Y[1:] - Y[:-1]) / dt
Y_mid = 0.5 * (Y[:-1] + Y[1:])

# Search for the hidden initial state
best = None
for x10 in torch.linspace(-1.0, 1.0, 101):
    x1 = reconstruct_x1(Y, x10)
    x1_mid = 0.5 * (x1[:-1] + x1[1:])
    theta = library(x1_mid, Y_mid, U)
    xi = sindy(theta, dY)
    error = torch.mean((theta @ xi - dY) ** 2)

    if best is None or error < best[0]:
        best = (error, x10, xi)

_, x10_hat, xi = best
x1_hat = reconstruct_x1(Y, x10_hat)
X_hat = torch.column_stack([x1_hat, Y])

# Results
names = [
    "1",
    "x1",
    "y",
    "u",
    "cos(x1+y)",
    "x1*cos(x1+y)",
]

print("\n" + "=" * 70)
print("DUFFING - SINDY WITH HIDDEN-STATE RECONSTRUCTION")
print("=" * 70)
print(f"Estimated x1(0): {x10_hat.item():.4f}")
print(f"Real x1(0):      {X_true[0, 0].item():.4f}")
print("\nIdentified equation: dot{y} =")
for name, coefficient in zip(names, xi):
    if abs(coefficient) >= threshold:
        print(f"  {coefficient.item():+.6f} * {name}")
print("=" * 70)

# Comparison of the real and reconstructed states
t = torch.arange(N + 1) * dt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(t.numpy(), X_true[:, 0].numpy(), label="Real x1")
plt.plot(t.numpy(), X_hat[:, 0].numpy(), "--", label="Estimated x1")
plt.title("Hidden state x1")
plt.xlabel("Time [s]")
plt.ylabel("x1")
plt.grid()
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t.numpy(), X_true[:, 1].numpy(), label="Real x2")
plt.plot(t.numpy(), X_hat[:, 1].numpy(), "--", label="Measured y = x2")
plt.title("Measured state x2")
plt.xlabel("Time [s]")
plt.ylabel("x2")
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()


## 11. Ruído de medição e perturbação de processo

O experimento foi ampliado para

\[
\dot x_2=-2x_2-x_1\cos(x_1+x_2)+u+w,
\]

\[
y=x_2+v.
\]

É essencial separar:
- w: perturbação de processo;
- v: ruído de medição.

### Por que o SINDy sofre com ruído de medição?
A diferenciação numérica amplifica alta frequência. Na aproximação simples,

\[
\dot y_k\approx\frac{y_{k+1}-y_k}{dt},
\]

o ganho de ruído está na escala de 1/dt. Com dt=0.01, essa escala é 100.

A reconstrução de x₁ usa integração, que tende a suavizar ruído. Por isso vimos casos em que a forma de x₁ permanecia plausível enquanto a equação identificada era contaminada.

### Achado principal
O SINDy consegue compensar:
- offset errado de x₁;
- ruído;
- perturbação;
- biblioteca incompleta;

alterando coeficientes e combinando termos.

Logo, **bom fit ou bom rollout de y não implica que o estado físico esteja correto**.

In [ ]:
# Duffing Oscillator - SINDy with measurement and process disturbances
# dot{x1} = x2
# dot{x2} = -delta*x2 - x1*cos(x1 + x2) + u + w
# y = x2 + v

import matplotlib.pyplot as plt
import torch

# For reproducibility purposes
torch.set_default_dtype(torch.float64)
torch.manual_seed(156)

# Parameters of simulation
dt = 0.01
T = 10.0
N = int(T / dt)
delta = 2.0
threshold = 0.1
process_noise = 0.5
parsimony_weight = 1e-4
noise_levels = [0.01, 0.03, 0.05, 0.10]

# Duffing oscillator dynamics, used only to simulate the plant
def dynamics(x, u, w):
    x1, x2 = x
    return torch.stack([
        x2,
        -delta * x2 - x1 * torch.cos(x1 + x2) + u + w,
    ])

# Simulation of one time step using RK4
def simulate_step(x, u, w):
    k1 = dynamics(x, u, w)
    k2 = dynamics(x + 0.5 * dt * k1, u, w)
    k3 = dynamics(x + 0.5 * dt * k2, u, w)
    k4 = dynamics(x + dt * k3, u, w)
    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

# Reconstruction from the known physics: dot{x1} = y
def reconstruct_x1(y, x10):
    x1 = torch.empty_like(y)
    x1[0] = x10
    x1[1:] = x10 + dt * torch.cumsum(
        0.5 * (y[:-1] + y[1:]),
        dim=0,
    )
    return x1

# Given that the physics is known, we can use basis physics, and the enhance with other candidates to explain
# Drawback is that the sindy WILL try to identify noise and disturbances as part of the dynamics, so it will be hard to identify the true dynamics if the noise is too high
# SINDy library containing the true nonlinear structure
def structured_library(x1, y, u):
    s = x1 + y
    return torch.stack([
        torch.ones_like(y),
        x1,
        y,
        u,
        torch.cos(s),
        x1 * torch.cos(s),
    ], dim=-1)

# SINDy library without the true nonlinear term
def blind_library(x1, y, u):
    s = x1 + y
    return torch.stack([
        torch.ones_like(y),
        y,
        u,
        x1**2,
        y**2,
        x1 * y,
        torch.sin(s),
        torch.cos(s),
        x1 * torch.sin(s),
    ], dim=-1)

# Sequential thresholded least squares
def sindy(theta, target):
    xi = torch.linalg.lstsq(theta, target).solution
    for _ in range(10):
        active = torch.abs(xi) >= threshold
        if not torch.any(active):
            break
        new_xi = torch.zeros_like(xi)
        new_xi[active] = torch.linalg.lstsq(theta[:, active], target).solution
        xi = new_xi
    return xi

# Simulation of the identified SINDy model
def simulate_sindy(x, u, xi, library):
    X = [x]

    for uk in u:
        dx2 = library(x[0], x[1], uk) @ xi
        x = x + dt * torch.stack([x[1], dx2])

        if not torch.all(torch.isfinite(x)) or torch.max(torch.abs(x)) > 10:
            x = torch.full_like(x, float("nan"))

        X.append(x)

    return torch.stack(X)

# SINDy identification and hidden-state reconstruction
def identify(y, u, library):
    dy = (y[2:] - y[:-2]) / (2.0 * dt)
    best = None

    for x10 in torch.linspace(-3.0, 3.0, 101):
        x1 = reconstruct_x1(y, x10)
        theta = library(x1[1:-1], y[1:-1], u[1:])
        xi = sindy(theta, dy)

        error = torch.mean((theta @ xi - dy) ** 2)
        score = error + parsimony_weight * torch.sum(torch.abs(xi))

        if best is None or score < best[0]:
            best = (score, x10, xi)

    x10_hat, xi = best[1], best[2]
    x1_hat = reconstruct_x1(y, x10_hat)
    X_sindy = simulate_sindy(
        torch.stack([x10_hat, y[0]]),
        u,
        xi,
        library,
    )

    return x10_hat, x1_hat, xi, X_sindy

# Real trajectory with process disturbance
x0 = torch.tensor([-0.6, 1.4])
U = 0.8 * torch.sin(torch.arange(N) * 0.2)
W = process_noise * torch.randn(N)

X_true = [x0]
for u, w in zip(U, W):
    X_true.append(simulate_step(X_true[-1], u, w))
X_true = torch.stack(X_true)
Y_true = X_true[:, 1]

# Noisy measurements
measurements = {
    noise: Y_true + noise * torch.std(Y_true) * torch.randn_like(Y_true)
    for noise in noise_levels
}

structured_results = {}
blind_results = {}

for noise, Y in measurements.items():
    structured_results[noise] = identify(Y, U, structured_library)
    blind_results[noise] = identify(Y, U, blind_library)

# State reconstruction errors
print("\n" + "=" * 84)
print("DUFFING - SINDY WITH DISTURBANCES")
print("=" * 84)
print("Noise    Structured x1(0)    Structured RMSE    Blind x1(0)    Blind RMSE")

for noise in noise_levels:
    structured_rmse = torch.sqrt(torch.mean(
        (structured_results[noise][1] - X_true[:, 0])**2
    ))
    blind_rmse = torch.sqrt(torch.mean(
        (blind_results[noise][1] - X_true[:, 0])**2
    ))

    print(
        f"{100 * noise:>4.0f}%"
        f"{structured_results[noise][0].item():>19.4f}"
        f"{structured_rmse.item():>19.6f}"
        f"{blind_results[noise][0].item():>15.4f}"
        f"{blind_rmse.item():>14.6f}"
    )

print("=" * 84)

# Identified equations for the 5% measurement-noise case
structured_names = [
    "1", "x1", "y", "u", "cos(x1+y)", "x1*cos(x1+y)"
]
blind_names = [
    "1", "y", "u", "x1^2", "y^2", "x1*y",
    "sin(x1+y)", "cos(x1+y)", "x1*sin(x1+y)"
]

for title, result, names in [
    ("Structured library", structured_results[0.05], structured_names),
    ("Blind library", blind_results[0.05], blind_names),
]:
    print(f"\n{title} - 5% measurement noise")
    print(f"Estimated x1(0): {result[0].item():.4f}")
    print("dot{y} =")

    for name, coefficient in zip(names, result[2]):
        if abs(coefficient) >= threshold:
            print(f"  {coefficient.item():+.6f} * {name}")

# Comparison of the real, measured and identified states
t = torch.arange(N + 1) * dt

def plot_result(noise, results, title):
    Y = measurements[noise]
    x1_hat = results[noise][1]
    X_sindy = results[noise][3]

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(t.numpy(), X_true[:, 0].numpy(), label="Real x1")
    plt.plot(t.numpy(), x1_hat.numpy(), "--", label="Estimated x1")
    plt.title(f"{title} - hidden state x1 - {100 * noise:.0f}% noise")
    plt.xlabel("Time [s]")
    plt.ylabel("x1")
    plt.grid()
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(t.numpy(), X_true[:, 1].numpy(), label="Real x2")
    plt.plot(t.numpy(), Y.numpy(), "--", label="Measured y")
    plt.plot(t.numpy(), X_sindy[:, 1].numpy(), ":", label="SINDy model")
    plt.title(f"{title} - state x2 - {100 * noise:.0f}% noise")
    plt.xlabel("Time [s]")
    plt.ylabel("x2")
    plt.grid()
    plt.legend()

    plt.tight_layout()
    plt.show()

# Structured library
plot_result(0.01, structured_results, "Structured SINDy")
# plot_result(0.03, structured_results, "Structured SINDy")
# plot_result(0.05, structured_results, "Structured SINDy")
plot_result(0.10, structured_results, "Structured SINDy")

# Blind library
plot_result(0.01, blind_results, "Blind SINDy")
# plot_result(0.03, blind_results, "Blind SINDy")
# plot_result(0.05, blind_results, "Blind SINDy")
plot_result(0.10, blind_results, "Blind SINDy")


## 12. O problema é também de identificabilidade

A equação \dot x₁=y determina x₁ somente até uma constante:

\[
x_1(t)=c+\int_0^t y(\tau)d\tau.
\]

No caso ideal, a segunda equação e uma biblioteca correta podem selecionar c. Com ruído ou biblioteca flexível, diferentes offsets podem ser acomodados por diferentes coeficientes SINDy.

Assim, reconstruir um estado físico oculto exige mais do que ajustar entrada-saída.

### Próximos testes já identificados
- selecionar x₁(0) por erro de rollout, não apenas erro local em \dot y;
- weak/integral SINDy para evitar derivação direta;
- várias trajetórias com dinâmica comum;
- calibrações esparsas do estado físico;
- SINDyc explícito;
- usar os termos encontrados pelo SINDy para orientar um lifting Koopman mais interpretável.

## 13. Sistemas instáveis

Koopman não exige que a planta seja estável. Se o sistema verdadeiro tem modos instáveis, o modelo identificado deve poder representá-los.

Consequências:
- impor ρ(A)<1 durante a identificação cria viés se a planta aberta é instável;
- trajetórias longas podem escapar rapidamente;
- ensaios curtos podem capturar dinâmica local;
- operar em malha fechada pode manter a planta em região informativa;
- mas dados em malha fechada exigem cuidado com identificação sob feedback.

A pergunta correta é:

> **Quais propriedades pertencem à planta e devem ser preservadas no modelo, e quais propriedades pertencem ao controlador e devem ser impostas depois?**

## 14. Koopman para EDPs

Uma EDP já possui um estado infinito-dimensional: por exemplo, a concentração c(r,t) é uma função do espaço.

O operador de Koopman continua conceitualmente aplicável, mas a implementação precisa reduzir o problema. Rotas discutidas:
1. discretização espacial: EDP → muitas EDOs;
2. redução modal/POD;
3. Koopman sobre estados discretizados ou coordenadas modais;
4. lifting estruturado;
5. observador/controlador no modelo reduzido.

Há duas explosões de dimensão:
- ordem da discretização da EDP;
- ordem do lifting de Koopman.

Por isso, para aplicações físicas, a pergunta passa a ser qual representação de baixa ordem preserva informação relevante para estimação e controle.

In [ ]:
# Demonstração didática: EDP de difusão 1D -> sistema de EDOs.
# Não é o modelo eletroquímico KPR; serve apenas para visualizar
# crescimento de dimensão e rigidez com a discretização espacial.

import numpy as np
import matplotlib.pyplot as plt

def diffusion_matrix(n, alpha=1.0, length=1.0):
    dx = length/(n+1)
    main = -2*np.ones(n)
    off = np.ones(n-1)
    L = np.diag(main) + np.diag(off, 1) + np.diag(off, -1)
    return alpha*L/(dx**2)

plt.figure(figsize=(8,4))
for n in [8,16,32]:
    eig = np.linalg.eigvals(diffusion_matrix(n))
    plt.scatter(np.full(n,n), eig.real, s=16, label=f"n={n}")
    print(
        f"n={n:2d} | estado={n:2d} | "
        f"modo mais lento={eig.real.max():.3f} | "
        f"modo mais rápido={eig.real.min():.3f}"
    )

plt.xlabel("ordem da discretização")
plt.ylabel("parte real dos autovalores")
plt.title("EDP -> EDO: dimensão e rigidez crescem com a discretização")
plt.grid()
plt.legend()
plt.show()


## 15. Ponte para baterias — KPR-25

O trabalho de Khalil, Postoyan e Raël parte de EDPs de difusão de lítio nos eletrodos e usa discretização espacial para obter um modelo de estado finito-dimensional.

O problema é diretamente conectado ao que discutimos para Koopman em EDPs: modelos de alta ordem representam melhor a física, mas elevam o custo de observadores e implementação.

A contribuição do KPR-25 corrige as concentrações geradas pelos modelos finito-dimensionais para que, sob corrente constante, tendam assintoticamente às concentrações da EDP original. Em seguida, são projetados observadores com análise de Lyapunov e validação experimental.

A cadeia é:

\[
\text{EDP física}
\rightarrow
\text{modelo finito-dimensional}
\rightarrow
\text{correção estruturada}
\rightarrow
\text{observador com garantia}.
\]

Isso é muito próximo do objetivo desta pesquisa: **reduzir/aprender sem perder a estrutura necessária para estimação e controle**.

## 16. KPR-25b — Luenberger não linear e LMI

O artigo considera uma classe ampla de modelos de bateria:

\[
\dot x=Ax+\psi(u,y)+Ew
\]

\[
y=Cx+\phi_1(x)+\phi_2(u)+Fv.
\]

É proposto um observador não linear Luenberger-like com ganho constante calculado offline por LMI.

O ponto mais relevante para este projeto é que a LMI não aparece como um artifício numérico isolado: ela certifica convergência robusta/global do erro de estimação, no sentido de estabilidade exponencial input-to-state frente a perturbação e ruído.

Isso conecta diretamente dois blocos que até então apareciam separados:
- identificação/redução do modelo;
- projeto de observador com garantias.

Uma direção natural é perguntar se um modelo aprendido/reduzido por Koopman/SINDy pode preservar a estrutura necessária para esse tipo de observador.

## 17. KPRN-26 — packs e estimação híbrida

O trabalho posterior sobre SOC mínimo/máximo em packs reforça outra ideia importante: estrutura híbrida e Lyapunov podem resolver escalabilidade sem tornar todo o problema black-box.

O estimador proposto tem dimensão independente do número de células e usa seleção dinâmica das células candidatas aos extremos de SOC, com garantias robustas via ferramentas de Lyapunov não suave.

Para esta pesquisa, a lição é metodológica: **redução de complexidade pode vir de estrutura matemática**, não apenas de redes neurais.

## 18. O que pode virar contribuição de dissertação

Os experimentos convergem para quatro eixos.

### Representação orientada a controle
- Koopman com observáveis interpretáveis;
- SINDy para descobrir estrutura;
- física parcial para ancorar estados;
- redução de EDPs preservando variáveis relevantes.

### Garantias
- Lyapunov/LMI durante ou após identificação;
- dissipatividade e propriedades físicas;
- observadores com convergência;
- robustez a ruído e mismatch.

### Identificabilidade
- previsão não é realização física;
- delays recuperam memória, não necessariamente semântica;
- estado oculto precisa de física, calibração ou estrutura.

### Baterias
- EDP → modelo finito-dimensional;
- SOC/concentração como estados não medidos;
- degradação/variação temporal distinta da não linearidade intrínseca;
- observação robusta como requisito natural.

Hipótese de trabalho que hoje parece coerente:

> **usar identificação estruturada — SINDy, Koopman e grey-box — para obter modelos compactos e interpretáveis, conectando-os depois a observadores/controladores com garantias Lyapunov/LMI.**

## 19. Backlog experimental

### Koopman
- manter LS como baseline explícito;
- sweep de RBFs/dimensão do lifting;
- sweep de número e comprimento de trajetórias;
- cobertura e condicionamento da regressão;
- ruído de medição e processo separados;
- one-step vs rollout;
- sistema realmente instável sem estabilidade artificial.

### SINDy
- weak/integral SINDy;
- SINDyc;
- múltiplas trajetórias, dinâmica compartilhada;
- x₁(0) por rollout;
- biblioteca progressivamente menos informada;
- separação entre parcimônia e identificabilidade.

### Observadores
- Luenberger em lifting que preserve estados físicos;
- observabilidade/detectabilidade;
- estado lifted vs estado físico;
- depois comparação com Kalman/robusto.

### EDP/bateria
- reproduzir discretização/correção do KPR-25;
- mapear estados medidos, não medidos, conhecidos e identificados;
- avaliar Koopman/SINDy sobre coordenadas reduzidas;
- testar se correção aprendida preserva condições necessárias ao observador.

## 20. Critérios de avaliação

Um método não deve ser considerado melhor apenas por reduzir RMSE.

### Modelo
- MAE/RMSE/IAE one-step e rollout;
- generalização fora da região de treino;
- ruído e quantidade de dados.

### Estrutura
- dimensão;
- número de termos ativos;
- interpretabilidade;
- condicionamento;
- estabilidade/dissipatividade/física.

### Estimação
- erro nos estados físicos;
- convergência;
- robustez;
- necessidade de sensores/calibrações.

### Controle
- possibilidade de síntese;
- garantias;
- custo online;
- robustez a mismatch.

### Computação
- tempo de identificação;
- tempo de atualização;
- custo por passo;
- escalabilidade.

## 21. Armadilhas que já identificamos

1. Confundir predição com controle.
2. Confundir estabilidade do modelo Koopman com estabilidade da planta.
3. Forçar estabilidade sobre uma dinâmica verdadeiramente instável.
4. Avaliar apenas one-step.
5. Confundir estado de realização I/O com estado físico.
6. Diferenciar diretamente sinais ruidosos sem avaliar amplificação.
7. Permitir que uma biblioteca SINDy flexível compense estado errado.
8. Comparar formulações com funções objetivo diferentes como se fossem idênticas.
9. Aumentar lifting sem medir conditioning/custo.
10. Ir cedo demais para redes antes de fechar os baselines estruturados.

## 22. Mapa dos métodos

\[
\text{dados + física}
\rightarrow
\begin{cases}
\text{SINDy: descoberta esparsa}\\
\text{Koopman: representação linear em observáveis}
\end{cases}
\]

\[
\rightarrow
\text{modelo orientado a controle}
\rightarrow
\begin{cases}
\text{Lyapunov / LMI}\\
\text{observadores}
\end{cases}
\rightarrow
\text{controle / baterias}.
\]

A direção mais promissora não parece ser escolher um vencedor único, mas usar cada ferramenta no ponto onde sua estrutura é mais valiosa.

## 23. Referências essenciais

1. **Sivaranjani et al. (2026)** — Control-Oriented System Identification: Classical, Learning, and Physics-Informed Approaches. Annual Reviews in Control 62, 101067. DOI 10.1016/j.arcontrol.2026.101067.  
   Papel: mapa conceitual; hard/soft constraints; Koopman; SINDy; física e propriedades de controle.

2. **Brunton, Proctor & Kutz (2016)** — Discovering governing equations from data by sparse identification of nonlinear dynamical systems. PNAS 113(15), 3932–3937. DOI 10.1073/pnas.1517384113.  
   Papel: fundamento do SINDy.

3. **Brunton, Proctor & Kutz (2016)** — Sparse Identification of Nonlinear Dynamics with Control. IFAC-PapersOnLine 49(18), 710–715. DOI 10.1016/j.ifacol.2016.10.249.  
   Papel: SINDyc.

4. **Brunton et al. (2016)** — Koopman Invariant Subspaces and Finite Linear Representations of Nonlinear Dynamical Systems for Control. PLOS ONE 11(2), e0150171. DOI 10.1371/journal.pone.0150171.  
   Papel: representações finitas e controle.

5. **Khalil, Postoyan & Raël (2025)** — Enhancing Accuracy of Finite-Dimensional Models for Lithium-Ion Batteries, Observer Design, and Experimental Validation. IEEE TCST 33(1), 327–342. DOI 10.1109/TCST.2024.3473769.  
   Papel: EDP → modelo finito → correção → observador.

6. **Khalil, Postoyan & Raël (2025)** — Systematic Observer Design With Robust Global Convergence Guarantees for a Large Class of Lithium-Ion Battery Models. IEEE Control Systems Letters 9. DOI 10.1109/LCSYS.2025.3575681.  
   Papel: Luenberger-like + LMI + convergência robusta/global.

7. **Khalil, Postoyan, Raël et al. (2026)** — Estimation of the minimum and maximum states of charge of lithium-ion battery packs: A hybrid approach. Automatica 183, 112630. DOI 10.1016/j.automatica.2025.112630.  
   Papel: estimação híbrida e escalabilidade.

### Referências conceituais já discutidas
Dawson et al. (neural Lyapunov-barrier), Wabersich & Zeilinger (predictive safety filter), Hu et al. (RNN online + LMPC), Salzmann et al. (residual neural model + real-time MPC), GP-MPC online, Neural ODE/KNODE e Port-Hamiltonian.

## 24. Decisões que não devem se perder

- Não pular direto para Deep Koopman.
- Fechar baseline clássico antes de redes.
- A pesquisa é control-oriented, não apenas prediction-oriented.
- Estado físico importa quando o objetivo é observação/controle.
- Física parcial deve ser usada como âncora quando disponível.
- Ruído de medição e processo devem ser isolados.
- Sistemas instáveis exigem tratamento próprio.
- Degradação de bateria é variação temporal/paramétrica e não deve ser confundida com não linearidade intrínseca.
- LMI/Lyapunov continua sendo ferramenta central.
- O GitHub deve registrar resultados positivos, limitações e tentativas que falharam.

## 25. Pergunta operacional atual

> **Como construir uma representação de baixa ordem, identificada a partir de dados e física parcial, que preserve significado suficiente para estimação e permita impor ou usar garantias Lyapunov-LMI sem sacrificar demais a fidelidade dinâmica?**

Essa pergunta conecta Koopman, SINDy, estado oculto, observadores, EDPs, baterias e robustez.

Sequência natural:

\[
\text{Duffing}
\rightarrow
\text{estado oculto + ruído}
\rightarrow
\text{estrutura física}
\rightarrow
\text{observador}
\rightarrow
\text{modelo reduzido de bateria}
\rightarrow
\text{garantias}.
\]

## 26. Como atualizar este memorando

A cada nova rodada, registrar:
1. **Pergunta:** qual limitação estamos atacando?
2. **Mudança:** o que mudou no método/código?
3. **Resultado:** o que melhorou, piorou ou ficou ambíguo?
4. **Interpretação:** isso muda a direção ou apenas o tuning?

A cadeia a preservar é:

\[
\text{hipótese}
\rightarrow
\text{experimento}
\rightarrow
\text{evidência}
\rightarrow
\text{decisão}.
\]

### Scripts de referência atuais
- systems/duffing_oscillator/classic_koopman.py
- systems/duffing_oscillator/koopman_via_lmis.py
- systems/duffing_oscillator/input_output_identification.py
- systems/duffing_oscillator/SINDy.py
- systems/duffing_oscillator/SINDy_with_disturbances.py

Este notebook deve ser atualizado quando uma descoberta alterar a interpretação do projeto, não apenas quando um parâmetro mudar.